In [143]:
import pandas as pd
import sys

sys.path.insert(0, '..') # Config-driven
from config import settings

report = pd.read_csv(settings.OUTPUT_DIR / 'validation_report.csv')
clean = pd.read_csv(settings.OUTPUT_DIR / 'valid_transactions.csv')
flagged = pd.read_csv(settings.OUTPUT_DIR / 'flagged_issues.csv')

print(f"Report:  {report.shape}")
print(f"Clean:   {clean.shape}")
print(f"Flagged: {flagged.shape}")

Report:  (200, 49)
Clean:   (72, 49)
Flagged: (128, 49)


In [144]:
report.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 49 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   transaction_id            200 non-null    str  
 1   lc_number                 198 non-null    str  
 2   lc_form                   200 non-null    str  
 3   available_with            200 non-null    str  
 4   issue_date                198 non-null    str  
 5   expiry_date               200 non-null    str  
 6   latest_shipment_date      200 non-null    str  
 7   applicant_name            199 non-null    str  
 8   applicant_address         198 non-null    str  
 9   applicant_city            200 non-null    str  
 10  applicant_postal_code     200 non-null    str  
 11  applicant_country         200 non-null    str  
 12  applicant_lei             199 non-null    str  
 13  applicant_account         200 non-null    str  
 14  beneficiary_name          200 non-null    str  
 15  

In [145]:
report.describe()


,tolerance_percent,presentation_period_days,error_count
count,200.000000,200.000000,200.000000
mean,4.775000,17.745000,1.185000
std,2.978039,3.500176,1.318891
min,0.000000,14.000000,0.000000
25%,5.000000,14.000000,0.000000
50%,5.000000,21.000000,1.000000
75%,5.000000,21.000000,2.000000
max,10.000000,21.000000,6.000000


In [146]:
# How many unique values per column?
print("=== UNIQUE VALUES ===")
print(report.nunique().sort_values(ascending=False).head(10))

# Which transactions have the most errors?
print("\n=== MOST PROBLEMATIC TRANSACTIONS ===")
print(report.nlargest(5, 'error_count')[['transaction_id', 'error_count', 'error_codes', 'severity']])


=== UNIQUE VALUES ===
transaction_id          200
lc_number               196
amount                  187
beneficiary_account     100
applicant_account       100
vessel_name             100
error_messages          100
error_codes              78
latest_shipment_date     72
expiry_date              67
dtype: int64

=== MOST PROBLEMATIC TRANSACTIONS ===
    transaction_id  error_count  \
44       LC-JZA7TZ            6   
71       LC-D84U89            6   
87       LC-CW790P            5   
96       LC-ZE5R5U            5   
112      LC-L6246H            5   

                                          error_codes  severity  
44    AMT002, LEI004, LEI004, LC005, SHIP002, SHIP005  CRITICAL  
71   AMT002, LEI004, LC001, SHIP002, SHIP005, XVAL002  CRITICAL  
87         LEI004, DATE005, SHIP002, SHIP005, XVAL002  CRITICAL  
96          AMT002, LEI004, SHIP002, XVAL001, XVAL002  CRITICAL  
112           AMT005, LEI004, LC003, SHIP002, SHIP005  CRITICAL  


In [147]:
# Quick profile of key categorical columns
for col in ['currency', 'severity', 'validation_status', 'incoterm', 'lc_form']:
    print(f"\n{col}:")
    print(report[col].value_counts())


currency:
currency
JPY    58
USD    51
GBP    47
EUR    44
Name: count, dtype: int64

severity:
severity
HIGH        77
NONE        72
CRITICAL    39
MEDIUM      11
LOW          1
Name: count, dtype: int64

validation_status:
validation_status
FLAGGED    128
CLEAN       72
Name: count, dtype: int64

incoterm:
incoterm
CPT    24
FCA    22
DAP    20
DDP    19
CFR    19
EXW    18
DPU    18
FAS    17
CIF    16
FOB    15
CIP     9
XXX     3
Name: count, dtype: int64

lc_form:
lc_form
STANDBY                     75
IRREVOCABLE                 66
IRREVOCABLE TRANSFERABLE    58
INVALID_FORM                 1
Name: count, dtype: int64


 Boolean Indexing! The core idea is simple: you ask pandas a yes/no question, and it gives you only the rows that say "yes."

In [148]:
# What does a boolean condition actually return?
report['currency'] == 'USD'

# True = show this row, False = hide it.

0      False
1      False
2       True
3       True
4      False
       ...  
195     True
196    False
197     True
198    False
199     True
Name: currency, Length: 200, dtype: bool

In [149]:
# Pass the mask inside brackets — only True rows survive
usd_transactions = report[report['currency'] == 'USD']
print(f"USD transactions: {len(usd_transactions)} out of {len(report)}")

USD transactions: 51 out of 200


In [150]:
# CRITICAL USD transactions
critical_usd = report[(report['currency'] == 'USD') & (report['severity'] == 'CRITICAL')]
print(f"Critical USD: {len(critical_usd)}")

# Flagged but NOT critical
flagged_not_critical = report[(report['validation_status'] == 'FLAGGED') & (report['severity'] != 'CRITICAL')]
print(f"Flagged but not critical: {len(flagged_not_critical)}")

# USD or EUR
usd_or_eur = report[(report['currency'] == "USD") | (report['currency'] == 'EUR')]
print(f"USD or EUR: {len(usd_or_eur)}")

Critical USD: 13
Flagged but not critical: 89
USD or EUR: 95


In [151]:
 # short way
usd_or_eur = report[report['currency'].isin(['USD', 'EUR'])]
report[report['currency'].isin(['USD', 'EUR', 'JPY'])]  # auto-displays- print()


,transaction_id,lc_number,lc_form,available_with,issue_date,expiry_date,latest_shipment_date,applicant_name,applicant_address,applicant_city,...,documents_required,additional_conditions,presentation_period_days,payment_terms,charges,validation_status,error_count,error_codes,error_messages,severity
1,LC-OH9SDB,LC-2026-148,STANDBY,BY ACCEPTANCE,2026-01-20,2026-05-23,2026-05-05,Apple Inc.,One Apple Park Way,Cupertino,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,60 DAYS AFTER SIGHT,SHA,CLEAN,0,NaN,NaN,NONE
2,LC-6GVWRD,LC-2026-138,STANDBY,BY DEFERRED PAYMENT,2026-01-16,2026-07-04,2026-06-14,Toyota Motor Credit,6565 Headquarters Dr,Plano,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,SHA,CLEAN,0,NaN,NaN,NONE
3,LC-0OXFQ8,LC-2026-084,STANDBY,BY MIXED PAYMENT,2026-01-10,2028-06-10,2026-05-23,Samsung Electronics,129 Samsung-ro,Suwon,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,60 DAYS AFTER SIGHT,OUR,FLAGGED,2,"LEI004, DATE007",[HIGH] LEI004: applicant_lei - LEI not found i...,HIGH
5,LC-L2W37D,LC-2026-110,STANDBY,BY MIXED PAYMENT,2026-01-23,2026-07-05,2026-06-18,Sony Corporation,1-7-1 Konan,Tokyo,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,AT SIGHT,BEN,CLEAN,0,NaN,NaN,NONE
6,LC-5LXO6Q,LC-2026-194,IRREVOCABLE TRANSFERABLE,BY NEGOTIATION,2026-01-09,2026-06-14,2026-05-26,LG Electronics,128 Yeoui-daero,Seoul,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,AT SIGHT,OUR,FLAGGED,1,LEI004,[HIGH] LEI004: applicant_lei - LEI not found i...,HIGH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,LC-TD96QE,LC-2026-153,STANDBY,BY NEGOTIATION,2026-01-11,2026-05-24,2026-05-01,Nestle SA,Avenue Nestle 55,Vevey,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,OUR,FLAGGED,1,XVAL002,[MEDIUM] XVAL002: issuing_bank_country - Issui...,MEDIUM
196,LC-CVCJWT,LC-2026-028,STANDBY,BY PAYMENT,2026-01-08,2026-05-24,2026-04-30,Microsoft Corporation,NaN,Redmond,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,AT SIGHT,BEN,FLAGGED,1,PTY004,[HIGH] PTY004: applicant_address - Party addre...,HIGH
197,LC-3DD03U,LC-2026-043,IRREVOCABLE TRANSFERABLE,BY NEGOTIATION,2026-01-18,2026-06-14,2026-05-27,Samsung Electronics,129 Samsung-ro,Suwon,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,AT SIGHT,OUR,FLAGGED,1,SHIP004,[MEDIUM] SHIP004: port_of_loading - Invalid po...,MEDIUM
198,LC-9XPSEI,LC-2026-185,IRREVOCABLE TRANSFERABLE,BY PAYMENT,2026-01-23,2026-06-19,2026-05-31,Apple Inc.,One Apple Park Way,Cupertino,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,AT SIGHT,SHA,CLEAN,0,NaN,NaN,NONE


In [152]:
# isin() — much cleaner for multiple values
major_currencies = report[report['currency'].isin(['USD', 'EUR', 'GBP'])]
print(f"Major currencies: {len(major_currencies)}")

# Combine isin with another condition
critical_major = report[
    report['currency'].isin(['USD', 'EUR', 'GBP']) &
    (report['severity'] == 'CRITICAL')
    ]
print(f"Critical in major currencies: {len(critical_major)}")

Major currencies: 142
Critical in major currencies: 28


between() — perfect for numeric ranges:

In [153]:
# Transactions with 3-5 errors
multi_error = report[report['error_count'].between(3,5)] # inclusive on both ends
print(f"3-5 errors: {len(multi_error)}")
print(multi_error[['transaction_id', 'error_count', 'error_codes']])

3-5 errors: 23
    transaction_id  error_count                                 error_codes
28       LC-P0TVHH            4           LEI004, DATE005, SHIP002, SHIP005
43       LC-ID25OW            3                    LEI004, SHIP002, SHIP006
61       LC-I3QK2I            3                     LEI004, LEI004, XVAL002
82       LC-DZR1YX            3                       LEI004, BIC001, LC007
83       LC-UUZPEA            4            AMT002, LEI004, SHIP002, SHIP005
87       LC-CW790P            5  LEI004, DATE005, SHIP002, SHIP005, XVAL002
90       LC-HE7UR2            4              AMT002, LEI004, LC002, SHIP002
96       LC-ZE5R5U            5   AMT002, LEI004, SHIP002, XVAL001, XVAL002
97       LC-NAM148            4            AMT001, LEI004, SHIP002, SHIP005
100      LC-U2GC5W            3                    LEI004, XVAL001, XVAL002
103      LC-ZP08J3            3                     AMT002, LEI004, SHIP002
106      LC-PUXQPH            4           LEI004, DATE004, DATE005, SHIP0

In [154]:
# query() — string-based filtering
# Same as: report[(report['error_count'] > 2) & (report['currency'] == 'JPY')]
result = report.query("error_count > 2 and currency == 'JPY'")
print(f"JPY with 3+ errors: {len(result)}")
print(result[['transaction_id', 'error_count', 'currency', 'severity']])


JPY with 3+ errors: 4
    transaction_id  error_count currency  severity
103      LC-ZP08J3            3      JPY  CRITICAL
111      LC-36KOF8            4      JPY  CRITICAL
133      LC-JLTEIY            3      JPY  CRITICAL
137      LC-A26YFP            3      JPY  CRITICAL



"Find all FLAGGED transactions in major currencies (USD, EUR, GBP) with 2 or more errors where severity is CRITICAL."



In [155]:
report.query("error_count > 2 and currency in ['GPB', 'USD', 'EUR'] and severity == 'CRITICAL' and validation_status == 'FLAGGED' ")


,transaction_id,lc_number,lc_form,available_with,issue_date,expiry_date,latest_shipment_date,applicant_name,applicant_address,applicant_city,...,documents_required,additional_conditions,presentation_period_days,payment_terms,charges,validation_status,error_count,error_codes,error_messages,severity
28,LC-P0TVHH,LC-2026-081,IRREVOCABLE,BY NEGOTIATION,2026-01-15,2026-06-15,2026-08-20,Toyota Motor Corporation,1 Toyota-cho,Toyota City,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,90 DAYS AFTER SIGHT,BEN,FLAGGED,4,"LEI004, DATE005, SHIP002, SHIP005",[HIGH] LEI004: applicant_lei - LEI not found i...,CRITICAL
83,LC-UUZPEA,LC-2026-095,IRREVOCABLE TRANSFERABLE,BY PAYMENT,2026-01-10,2026-05-16,2026-04-23,NVIDIA Corporation,2788 San Tomas Expy,Santa Clara,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,OUR,FLAGGED,4,"AMT002, LEI004, SHIP002, SHIP005",[CRITICAL] AMT002: amount - Amount negative (g...,CRITICAL
87,LC-CW790P,LC-2026-088,STANDBY,BY MIXED PAYMENT,2026-01-15,2026-06-15,2026-08-20,Nestle SA,Avenue Nestle 55,Vevey,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,SHA,FLAGGED,5,"LEI004, DATE005, SHIP002, SHIP005, XVAL002",[HIGH] LEI004: applicant_lei - LEI not found i...,CRITICAL
90,LC-HE7UR2,LC,IRREVOCABLE TRANSFERABLE,BY ACCEPTANCE,2026-01-11,2026-07-04,2026-06-04,Samsung Electronics,129 Samsung-ro,Suwon,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,90 DAYS AFTER SIGHT,SHA,FLAGGED,4,"AMT002, LEI004, LC002, SHIP002",[CRITICAL] AMT002: amount - Amount negative (g...,CRITICAL
96,LC-ZE5R5U,LC-2026-057,STANDBY,BY MIXED PAYMENT,2026-01-14,2026-07-07,2026-06-17,Nestle SA,Avenue Nestle 55,Vevey,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,90 DAYS AFTER SIGHT,BEN,FLAGGED,5,"AMT002, LEI004, SHIP002, XVAL001, XVAL002",[CRITICAL] AMT002: amount - Amount negative (g...,CRITICAL
106,LC-PUXQPH,LC-2026-092,IRREVOCABLE TRANSFERABLE,BY PAYMENT,2026-06-01,2026-03-01,2026-05-11,TSMC,8 Li-Hsin Rd,Hsinchu,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,AT SIGHT,BEN,FLAGGED,4,"LEI004, DATE004, DATE005, SHIP002",[HIGH] LEI004: applicant_lei - LEI not found i...,CRITICAL
185,LC-ZJJEGE,LC-2026-056,IRREVOCABLE,INVALID_OPTION,2026-01-20,2026-07-09,2026-06-11,Toyota Motor Corporation,1 Toyota-cho,Toyota City,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,90 DAYS AFTER SIGHT,SHA,FLAGGED,4,"LEI004, LEI004, LC005, SHIP002",[HIGH] LEI004: applicant_lei - LEI not found i...,CRITICAL
191,LC-3RZSJ4,LC-2026-099,STANDBY,BY MIXED PAYMENT,2026-01-12,2026-05-20,2026-04-22,BMW AG,Petuelring 130,Munich,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,AT SIGHT,BEN,FLAGGED,4,"AMT002, LEI004, SHIP002, SHIP005",[CRITICAL] AMT002: amount - Amount negative (g...,CRITICAL


In [156]:
report.query(
      "error_count > 2"
      " and currency in ['JPY', 'USD', 'EUR']"
      " and severity == 'CRITICAL'"
      " and validation_status == 'FLAGGED'"
  )

,transaction_id,lc_number,lc_form,available_with,issue_date,expiry_date,latest_shipment_date,applicant_name,applicant_address,applicant_city,...,documents_required,additional_conditions,presentation_period_days,payment_terms,charges,validation_status,error_count,error_codes,error_messages,severity
28,LC-P0TVHH,LC-2026-081,IRREVOCABLE,BY NEGOTIATION,2026-01-15,2026-06-15,2026-08-20,Toyota Motor Corporation,1 Toyota-cho,Toyota City,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,90 DAYS AFTER SIGHT,BEN,FLAGGED,4,"LEI004, DATE005, SHIP002, SHIP005",[HIGH] LEI004: applicant_lei - LEI not found i...,CRITICAL
83,LC-UUZPEA,LC-2026-095,IRREVOCABLE TRANSFERABLE,BY PAYMENT,2026-01-10,2026-05-16,2026-04-23,NVIDIA Corporation,2788 San Tomas Expy,Santa Clara,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,OUR,FLAGGED,4,"AMT002, LEI004, SHIP002, SHIP005",[CRITICAL] AMT002: amount - Amount negative (g...,CRITICAL
87,LC-CW790P,LC-2026-088,STANDBY,BY MIXED PAYMENT,2026-01-15,2026-06-15,2026-08-20,Nestle SA,Avenue Nestle 55,Vevey,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,SHA,FLAGGED,5,"LEI004, DATE005, SHIP002, SHIP005, XVAL002",[HIGH] LEI004: applicant_lei - LEI not found i...,CRITICAL
90,LC-HE7UR2,LC,IRREVOCABLE TRANSFERABLE,BY ACCEPTANCE,2026-01-11,2026-07-04,2026-06-04,Samsung Electronics,129 Samsung-ro,Suwon,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,90 DAYS AFTER SIGHT,SHA,FLAGGED,4,"AMT002, LEI004, LC002, SHIP002",[CRITICAL] AMT002: amount - Amount negative (g...,CRITICAL
96,LC-ZE5R5U,LC-2026-057,STANDBY,BY MIXED PAYMENT,2026-01-14,2026-07-07,2026-06-17,Nestle SA,Avenue Nestle 55,Vevey,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,90 DAYS AFTER SIGHT,BEN,FLAGGED,5,"AMT002, LEI004, SHIP002, XVAL001, XVAL002",[CRITICAL] AMT002: amount - Amount negative (g...,CRITICAL
103,LC-ZP08J3,LC-2026-068,IRREVOCABLE TRANSFERABLE,BY MIXED PAYMENT,2026-01-10,2026-06-16,2026-05-18,Company @#$% Inc,129 Samsung-ro,Suwon,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,90 DAYS AFTER SIGHT,OUR,FLAGGED,3,"AMT002, LEI004, SHIP002",[CRITICAL] AMT002: amount - Amount negative (g...,CRITICAL
106,LC-PUXQPH,LC-2026-092,IRREVOCABLE TRANSFERABLE,BY PAYMENT,2026-06-01,2026-03-01,2026-05-11,TSMC,8 Li-Hsin Rd,Hsinchu,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,AT SIGHT,BEN,FLAGGED,4,"LEI004, DATE004, DATE005, SHIP002",[HIGH] LEI004: applicant_lei - LEI not found i...,CRITICAL
111,LC-36KOF8,LC-2026-046,STANDBY,BY ACCEPTANCE,2026-01-20,2026-07-14,2026-06-17,LG Electronics,128 Yeoui-daero,Seoul,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,AT SIGHT,BEN,FLAGGED,4,"AMT005, LEI004, LEI004, XVAL001",[CRITICAL] AMT005: amount - Invalid amount for...,CRITICAL
133,LC-JLTEIY,LC-2026-083,IRREVOCABLE,BY DEFERRED PAYMENT,2026-01-09,2026-05-23,2026-04-25,LG Electronics,128 Yeoui-daero,Seoul,...,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,OUR,FLAGGED,3,"AMT005, LEI004, XVAL002",[CRITICAL] AMT005: amount - Invalid amount for...,CRITICAL
137,LC-A26YFP,LC-2026-061,IRREVOCABLE,BY NEGOTIATION,2026-01-17,2026-06-23,2026-05-27,HSBC USA Inc.,452 Fifth Ave,New York,...,Commercial Invoice; Packing List; Bill of Lading,NaN,21,60 DAYS AFTER SIGHT,OUR,FLAGGED,3,"LEI004, BIC001, LC007",[HIGH] LEI004: applicant_lei - LEI not found i...,CRITICAL


Group BY

In [157]:
# How many transactions per currency?
report.groupby('currency')['transaction_id'].count()

# groupby('currency') = split by currency.
# ['transaction_id'] = look at this column. .count() = count non-null values.

currency
EUR    44
GBP    47
JPY    58
USD    51
Name: transaction_id, dtype: int64

In [158]:
# Total errors per currency
report.groupby('currency')['error_count'].sum()

currency
EUR    40
GBP    67
JPY    58
USD    72
Name: error_count, dtype: int64

In [159]:
# Average errors per currency
report.groupby('currency')['error_count'].mean().round(2)

currency
EUR    0.91
GBP    1.43
JPY    1.00
USD    1.41
Name: error_count, dtype: float64

In [160]:
# Error count by currency AND severity
report.groupby(['currency', 'severity'])['transaction_id'].count()

currency  severity
EUR       CRITICAL     4
          HIGH        18
          MEDIUM       5
          NONE        17
GBP       CRITICAL    11
          HIGH        16
          LOW          1
          NONE        19
JPY       CRITICAL    11
          HIGH        24
          MEDIUM       2
          NONE        21
USD       CRITICAL    13
          HIGH        19
          MEDIUM       4
          NONE        15
Name: transaction_id, dtype: int64

In [161]:
# size() counts ALL rows (including NaN)
# count() counts only non-null values
print("size():")
print(report.groupby('severity').size()) # counts rows in the group — it doesn't need a column because it's counting the group itself
print("\ncount() on error_codes:")
print(report.groupby('severity')['error_codes'].count())  # counts non-null values in a specific column

size():
severity
CRITICAL    39
HIGH        77
LOW          1
MEDIUM      11
NONE        72
dtype: int64

count() on error_codes:
severity
CRITICAL    39
HIGH        77
LOW          1
MEDIUM      11
NONE         0
Name: error_codes, dtype: int64


In [162]:
# Multiple aggregations in one shot
report.groupby('currency')['error_count'].agg(['sum', 'mean', 'max', 'min'])

,sum,mean,max,min
currency,,,,
EUR,40,0.909091,4,0
GBP,67,1.425532,6,0
JPY,58,1.000000,4,0
USD,72,1.411765,5,0


# Named aggregations — you control the column names

In [163]:
# Named aggregations — you control the column names
report.groupby('currency').agg(
    total_transactions=('transaction_id', 'count'),
    total_errors=('error_count', 'sum'),
    avg_errors=('error_count', 'mean'),
    worst_case=('error_count', 'max')
).round(2)

,total_transactions,total_errors,avg_errors,worst_case
currency,,,,
EUR,44,40,0.91,4
GBP,47,67,1.43,6
JPY,58,58,1.00,4
USD,51,72,1.41,5


In [164]:
# Add a column: average error count for each currency group
report['currency_avg_errors'] = report.groupby('currency')['error_count'].transform('mean').round(2)

# Now each row has its currency's average alongside its own count
report[['transaction_id', 'currency', 'error_count', 'currency_avg_errors']].head(10)

,transaction_id,currency,error_count,currency_avg_errors
0,LC-ZCOTOH,GBP,0,1.43
1,LC-OH9SDB,JPY,0,1.00
2,LC-6GVWRD,USD,0,1.41
3,LC-0OXFQ8,USD,2,1.41
4,LC-Q2Y9V8,GBP,2,1.43
5,LC-L2W37D,EUR,0,0.91
6,LC-5LXO6Q,EUR,1,0.91
7,LC-TOGN1W,USD,0,1.41
8,LC-3J2D54,GBP,0,1.43
9,LC-V55PSQ,JPY,1,1.00


 "Which transactions have MORE errors than their currency average?

In [165]:
# Transactions above their currency average
above_avg = report[report['error_count'] > report['currency_avg_errors']]
print(f"Above their currency average: {len(above_avg)}")# Drop the temp column — we don't need it permanently

# Drop the temp column — we don't need it permanently
report.drop(columns='currency_avg_errors', inplace=True)

Above their currency average: 76


filter() — this one's different. It keeps or drops entire groups, not individual rows

In [166]:
# Keep only currencies that have more than 50 transactions
big_currencies = report.groupby('currency').filter(lambda g: len(g) > 50)
print(f"Original: {len(report)} rows")
print(f"After filter: {len(big_currencies)} rows")
print(f"Currencies kept: {big_currencies['currency'].unique()}")

Original: 200 rows
After filter: 109 rows
Currencies kept: <StringArray>
['JPY', 'USD']
Length: 2, dtype: str


# agg() on multiple columns

In [167]:
# agg() on multiple columns — different functions per column
report.groupby('currency').agg(
    total_txn=('transaction_id', 'count'),
    total_errors=('error_count', 'sum'),
    avg_errors=('error_count', 'mean'),
    avg_tolerance=('tolerance_percent', 'mean'),
    avg_presentation=('presentation_period_days', 'mean')
).round(2)


,total_txn,total_errors,avg_errors,avg_tolerance,avg_presentation
currency,,,,,
EUR,44,40,0.91,5.00,17.82
GBP,47,67,1.43,4.47,18.02
JPY,58,58,1.00,5.09,17.38
USD,51,72,1.41,4.51,17.84


In [168]:
# Two-level groupby with named agg
report.groupby(['currency', 'validation_status']).agg(
    count=('transaction_id', 'count'),
    total_errors=('error_count', 'sum'),
    max_errors=('error_count', 'max')  #  worst single transaction in the group
)

count  total_errors  max_errors
currency validation_status                                 
EUR      CLEAN                 17             0           0
         FLAGGED               27            40           4
GBP      CLEAN                 19             0           0
         FLAGGED               28            67           6
JPY      CLEAN                 21             0           0
         FLAGGED               37            58           4
USD      CLEAN                 15             0           0
         FLAGGED               36            72           5

transform() combined with groupby() for percentage calculations:

In [169]:
# What percentage of its currency's errors does each transaction represent?
report['pct_of_currency_errors'] = (  # Creating the percentage column
    report['error_count'] /
    report.groupby('currency')['error_count'].transform('sum') * 100
).round(2)

# Show flagged transactions with their percentage
# .query()  = report[report['validation_status'] == 'FLAGGED']
report.query("validation_status == 'FLAGGED'")[
    ['transaction_id', 'currency', 'error_count', 'pct_of_currency_errors']
].nlargest(10, 'error_count') # = .sort_values('error_count', ascending=False).head(10)

# "Among flagged transactions, which 10 have the most errors — and how much of their currency's total errors do they account for "



,transaction_id,currency,error_count,pct_of_currency_errors
44,LC-JZA7TZ,GBP,6,8.96
71,LC-D84U89,GBP,6,8.96
87,LC-CW790P,USD,5,6.94
96,LC-ZE5R5U,USD,5,6.94
112,LC-L6246H,GBP,5,7.46
116,LC-3WXA73,GBP,5,7.46
175,LC-0TPAC5,GBP,5,7.46
28,LC-P0TVHH,EUR,4,10.00
83,LC-UUZPEA,USD,4,5.56
90,LC-HE7UR2,USD,4,5.56


 filter() with groupby on aggregated conditions:

In [170]:
# Clean up temp column
report.drop(columns='pct_of_currency_errors', inplace=True)
# inplace=True means modify the original DataFrame directly instead of returning a new one.

# filter(): keep only currencies where average error count > 1.2
high_error_currencies = report.groupby('currency').filter(
    lambda  g: g['error_count'].mean() > 1.2
)
print(f"Currencies with avg errors > 1.2:")
print(high_error_currencies['currency'].unique())
print(f"Rows kept: {len(high_error_currencies)}")

Currencies with avg errors > 1.2:
<StringArray>
['GBP', 'USD']
Length: 2, dtype: str
Rows kept: 98


 groupby on a calculated/derived column

In [171]:
# Create error severity buckets and analyze
report['error_bucket'] = pd.cut(
    report['error_count'],
    bins=[-1, 0, 1, 2, 6],
    labels=['Clean', '1 error', '2 errors', '3+ errors']
)
# report['error_bucket'].value_counts()
report.groupby('error_bucket', observed=True).agg(
    count=('transaction_id', 'count'),
    avg_tolerance=('tolerance_percent', 'mean'),
    avg_presentation=('presentation_period_days', 'mean')
).round(2)
# observed=True means only show buckets that actually have data.

,count,avg_tolerance,avg_presentation
error_bucket,,,
Clean,72,5.00,17.99
1 error,69,4.49,17.75
2 errors,34,4.71,17.50
3+ errors,25,5.00,17.36


In [172]:
report.drop(columns='error_bucket', inplace=True)

Day 51 — Sorting & Ranking

In [173]:
# Sort by error count — worst first
report.sort_values('error_count', ascending=False).head(10)[
    ['transaction_id', 'currency', 'error_count', 'severity']
]
# ascending=False = biggest first. Same as nlargest(10, 'error_count')

,transaction_id,currency,error_count,severity
44,LC-JZA7TZ,GBP,6,CRITICAL
71,LC-D84U89,GBP,6,CRITICAL
96,LC-ZE5R5U,USD,5,CRITICAL
175,LC-0TPAC5,GBP,5,CRITICAL
116,LC-3WXA73,GBP,5,CRITICAL
112,LC-L6246H,GBP,5,CRITICAL
87,LC-CW790P,USD,5,CRITICAL
28,LC-P0TVHH,EUR,4,CRITICAL
106,LC-PUXQPH,USD,4,CRITICAL
90,LC-HE7UR2,USD,4,CRITICAL


 multi-column sorting — sort by two things at once

In [174]:
# Sort by severity first, then error count within each severity
severity_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3, 'NONE': 4}
report['severity_rank'] = report['severity'].map(severity_order)

report.sort_values(['severity_rank', 'error_count'], ascending=[True, False]).head(10)[
    ['transaction_id', 'severity', 'error_count', 'currency']
]

,transaction_id,severity,error_count,currency
44,LC-JZA7TZ,CRITICAL,6,GBP
71,LC-D84U89,CRITICAL,6,GBP
87,LC-CW790P,CRITICAL,5,USD
96,LC-ZE5R5U,CRITICAL,5,USD
112,LC-L6246H,CRITICAL,5,GBP
116,LC-3WXA73,CRITICAL,5,GBP
175,LC-0TPAC5,CRITICAL,5,GBP
28,LC-P0TVHH,CRITICAL,4,EUR
83,LC-UUZPEA,CRITICAL,4,USD
90,LC-HE7UR2,CRITICAL,4,USD


 rank() — it assigns a position to each row:

In [175]:
# Rank transactions by error count (highest = rank 1)
report['error_rank'] = report['error_count'].rank(ascending=False, method='dense').astype(int)
#  .rank() assigns a rank number to each row based on its value.

report.sort_values('error_rank').head(15)[
    ['transaction_id', 'error_count', 'error_rank', 'currency']
]

,transaction_id,error_count,error_rank,currency
44,LC-JZA7TZ,6,1,GBP
71,LC-D84U89,6,1,GBP
87,LC-CW790P,5,2,USD
112,LC-L6246H,5,2,GBP
96,LC-ZE5R5U,5,2,USD
116,LC-3WXA73,5,2,GBP
175,LC-0TPAC5,5,2,GBP
97,LC-NAM148,4,3,GBP
90,LC-HE7UR2,4,3,USD
83,LC-UUZPEA,4,3,USD


In [176]:
report.drop(columns=['severity_rank', 'error_rank'], inplace=True)